# 🧠 Assignment 5 — SimCLR: Self-Supervised Learning on CIFAR-10
## All Checkpoints 1 → 4

| Setting | Value |
|---------|-------|
| Dataset | CIFAR-10 |
| Seed | 2026 |
| Batch Size | 64 |
| Temperature τ | 0.5 |
| SimCLR Epochs | 50 |
| Linear Probe Epochs | 20 |
| Fine-tune Epochs | 20 |
| LR | 3e-4 |

**Runtime → GPU (T4). Run all cells top to bottom.**

---
## 0 — Environment & Setup

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
for d in ['data','splits','utils','results','graphs','models']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

In [ ]:
from google.colab import files
import shutil
print('Upload the 4 split .txt files from your instructor starter package ...')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.txt'):
        shutil.move(fname, f'splits/{fname}')
        print(f'  -> splits/{fname}')

## 0.1 — Utility Modules

In [ ]:
%%writefile utils/seed.py
import os, random
import numpy as np
import torch

def set_seed(seed=2026):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
%%writefile utils/dataset_splits.py
from pathlib import Path
from torch.utils.data import Dataset, Subset
from torchvision.datasets import CIFAR10

def read_split_indices(path):
    return [int(l.strip()) for l in Path(path).read_text().splitlines() if l.strip()]

def get_cifar10_subset(data_root, split_file, train, transform=None, target_transform=None, download=False):
    ds = CIFAR10(str(data_root), train=train, transform=transform,
                 target_transform=target_transform, download=download)
    return Subset(ds, read_split_indices(split_file))

class TwoViewDataset(Dataset):
    def __init__(self, base, two_view_tf):
        self.base = base
        self.tf = two_view_tf
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, lbl = self.base[i]
        v1, v2 = self.tf(img)
        return v1, v2, lbl

In [ ]:
%%writefile utils/metrics.py
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def save_confusion_matrix(y_true, y_pred, out_path, title='Confusion Matrix'):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(10)))
    fig, ax = plt.subplots(figsize=(8,8))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(title); fig.tight_layout()
    fig.savefig(out_path, dpi=150); plt.close(fig)
    print(f'Saved -> {out_path}')

In [ ]:
import sys; sys.path.insert(0,'.')
from utils.seed import set_seed
for f in ['train_labeled_10percent','train_ssl_unlabeled','val','test']:
    p = f'splits/{f}.txt'
    n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
    status = 'OK' if n > 0 else 'MISSING'
    print(f'  [{status}] {p}  ({n:,} lines)')

---
# CHECKPOINT 1 — Day 3
> Supervised Baseline · Augmentation Pipeline · TwoViewTransform · Visualisation

## 1.1 — Global Imports & Constants

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from utils.seed import set_seed
from utils.dataset_splits import get_cifar10_subset, TwoViewDataset
from utils.metrics import save_confusion_matrix

SEED = 2026
BATCH = 64
LR = 3e-4
TAU = 0.5
DATA_ROOT = './data'
SPLITS = './splits'
RESULTS = './results'
GRAPHS = './graphs'
MODELS = './models'
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'Device: {device}')

## 1.2 — ResNet-18 Modified for CIFAR-10

In [ ]:
def build_resnet18(num_classes=10):
    """
    ResNet-18 for CIFAR-10:
      conv1  : 7x7 stride-2 -> 3x3 stride-1, padding 1
      maxpool: replaced with Identity (no aggressive downsampling)
      fc     : 512 -> num_classes
    """
    m = torchvision.models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, num_classes)
    return m

# Sanity check
dummy = torch.zeros(2, 3, 32, 32)
print('Output shape:', build_resnet18()(dummy).shape, '  expected [2, 10]')

## 1.3 — Load Fixed Dataset Splits

In [ ]:
tr_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                   T.ToTensor(), T.Normalize(MEAN, STD)])
ev_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

tr_ds = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_labeled_10percent.txt',
                           train=True,  transform=tr_tf, download=True)
vl_ds = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/val.txt',  train=True,  transform=ev_tf)
te_ds = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/test.txt', train=False, transform=ev_tf)

kw = dict(batch_size=BATCH, num_workers=2, pin_memory=True)
tr_ldr = DataLoader(tr_ds, shuffle=True,  **kw)
vl_ldr = DataLoader(vl_ds, shuffle=False, **kw)
te_ldr = DataLoader(te_ds, shuffle=False, **kw)

print(f'Train (10% labeled): {len(tr_ds):,}')
print(f'Validation         : {len(vl_ds):,}')
print(f'Test               : {len(te_ds):,}')

## 1.4 — Supervised Baseline Training

In [ ]:
def train_one_epoch(model, loader, crit, opt, device):
    model.train()
    ls = co = tot = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward(); opt.step()
        ls  += loss.item() * x.size(0)
        co  += (out.argmax(1) == y).sum().item()
        tot += x.size(0)
    return ls / tot, co / tot

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    ls = co = tot = 0
    preds = []; labs = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss = crit(out, y)
        ls  += loss.item() * x.size(0)
        p    = out.argmax(1)
        co  += (p == y).sum().item()
        tot += x.size(0)
        preds.extend(p.cpu().tolist())
        labs.extend(y.cpu().tolist())
    return ls / tot, co / tot, preds, labs

In [ ]:
EPOCHS_SUP = 30
set_seed(SEED)

sup_model = build_resnet18().to(device)
crit      = nn.CrossEntropyLoss()
opt       = torch.optim.Adam(sup_model.parameters(), lr=LR)

tr_ls, vl_ls = [], []
best_acc, best_sd = 0.0, None

print(f'Training supervised baseline for {EPOCHS_SUP} epochs ...')
for ep in range(1, EPOCHS_SUP + 1):
    trl, tra         = train_one_epoch(sup_model, tr_ldr, crit, opt, device)
    vll, vla, _, _   = eval_epoch(sup_model, vl_ldr, crit, device)
    tr_ls.append(trl); vl_ls.append(vll)
    if vla > best_acc:
        best_acc = vla
        best_sd  = {k: v.clone() for k, v in sup_model.state_dict().items()}
    if ep % 5 == 0 or ep == 1:
        print(f'  Ep {ep:3d}/{EPOCHS_SUP}  tr_loss={trl:.4f} tr_acc={tra:.3f}  vl_loss={vll:.4f} vl_acc={vla:.3f}')

print(f'Best val acc: {best_acc:.4f}')

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, EPOCHS_SUP+1), tr_ls, lw=2, label='Train Loss')
ax.plot(range(1, EPOCHS_SUP+1), vl_ls, lw=2, label='Val Loss')
ax.set(xlabel='Epoch', ylabel='Cross-Entropy Loss',
       title='Supervised Baseline — Loss Curve (ResNet-18, 10% labels)')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{GRAPHS}/supervised_loss.png', dpi=150)
plt.show(); print(f'Saved -> {GRAPHS}/supervised_loss.png')

# Test evaluation
sup_model.load_state_dict(best_sd)
_, sup_test_acc, tp, tl = eval_epoch(sup_model, te_ldr, crit, device)
print(f'Supervised Test Accuracy: {sup_test_acc:.4f}')

save_confusion_matrix(tl, tp, f'{RESULTS}/supervised_confusion_matrix.png',
                      title='Supervised Baseline — Confusion Matrix')
torch.save(sup_model.state_dict(), f'{MODELS}/supervised_model.pt')
print(f'Saved -> {MODELS}/supervised_model.pt')

## 1.5 — SimCLR Augmentation Pipeline & TwoViewTransform

In [ ]:
# ---- SimCLR augmentation pipeline (exactly as specified in the assignment) ----
simclr_transform = T.Compose([
    T.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

plain_transform = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])


# ---- TwoViewTransform (implemented from scratch — no library) ----------------
class TwoViewTransform:
    """
    Applies `transform` twice independently to produce two differently-
    augmented views of the same PIL image.
    """
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        view1 = self.transform(x)
        view2 = self.transform(x)
        return view1, view2


# ---- Helper: reverse normalisation for display --------------------------------
def denorm(t):
    m = torch.tensor(MEAN).view(3, 1, 1)
    s = torch.tensor(STD).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)

# Quick test
from torchvision.datasets import CIFAR10
raw = CIFAR10('./data', train=True, transform=None, download=False)
v1, v2 = TwoViewTransform(simclr_transform)(raw[0][0])
print('View shape:', v1.shape, '   Identical:', torch.equal(v1, v2))

In [ ]:
set_seed(SEED)
raw_ds = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_labeled_10percent.txt',
                            train=True, transform=None)
tv = TwoViewTransform(simclr_transform)
N = 10

fig, axes = plt.subplots(N, 3, figsize=(9, N*2.5),
                          gridspec_kw={'wspace':0.05, 'hspace':0.4})
for col, title in enumerate(['Original', 'Augmented View 1', 'Augmented View 2']):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i in range(N):
    img_pil, label = raw_ds[i]
    v1, v2 = tv(img_pil)
    for col, t in enumerate([denorm(plain_transform(img_pil)), denorm(v1), denorm(v2)]):
        axes[i, col].imshow(t.permute(1,2,0).numpy())
        axes[i, col].axis('off')
    axes[i, 0].set_ylabel(CLASSES[label], fontsize=9, rotation=0, labelpad=44, va='center')

fig.suptitle('SimCLR Augmentation Examples', fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(f'{RESULTS}/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show(); print(f'Saved -> {RESULTS}/augmentation_examples.png')

print('\nTask 2 answers:')
print('Q1. Identical? No - each view is an independent random draw.')
print('Q2. Same object? Yes - augmentations are class-preserving.')
print('Q3. Why positive pair? Both come from the same image; good encoder embeds them nearby.')
print('Q4. Too weak? Views near-identical, task trivial, model learns little.')
print('Q5. Too strong? Views lose shared semantics; positives look like negatives.')

---
# CHECKPOINT 2 — Day 6
> Encoder · Projection Head · Pair Construction · Similarity Matrix · NT-Xent Loss

## 2.1 — Encoder (ResNet-18, CIFAR-10 modified, no pretraining)

In [ ]:
class Encoder(nn.Module):
    """
    ResNet-18 modified for CIFAR-10. Outputs 512-dim feature vector h.
      conv1   : 3x3 stride-1 padding-1  (replaces 7x7 stride-2)
      maxpool : Identity                 (no aggressive downsampling)
    """
    def __init__(self):
        super().__init__()
        bb = torchvision.models.resnet18(weights=None)
        bb.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        bb.maxpool = nn.Identity()
        self.features = nn.Sequential(
            bb.conv1, bb.bn1, bb.relu, bb.maxpool,
            bb.layer1, bb.layer2, bb.layer3, bb.layer4,
            bb.avgpool,
        )
    def forward(self, x):
        return self.features(x).flatten(1)   # (B, 512)

print('Encoder output:', Encoder()(torch.zeros(4, 3, 32, 32)).shape, '  expected [4, 512]')

## 2.2 — Projection Head & SimCLR Model

In [ ]:
class ProjectionHead(nn.Module):
    """
    Linear(512->256) -> ReLU -> Linear(256->128)
    Used ONLY during SimCLR pretraining. Downstream tasks use h, not z.
    """
    def __init__(self, in_dim=512, hid=256, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(inplace=True),
            nn.Linear(hid, out_dim),
        )
    def forward(self, h): return self.net(h)


class SimCLR(nn.Module):
    """Encoder + ProjectionHead. Returns projected representations z1, z2."""
    def __init__(self):
        super().__init__()
        self.encoder    = Encoder()
        self.projection = ProjectionHead()
    def forward(self, v1, v2):
        z1 = self.projection(self.encoder(v1))
        z2 = self.projection(self.encoder(v2))
        return z1, z2

m = SimCLR()
z1, z2 = m(torch.zeros(4,3,32,32), torch.zeros(4,3,32,32))
print('z1:', z1.shape, '  z2:', z2.shape, '  expected [4, 128]')

## 2.3 — Positive / Negative Pair Table

In [ ]:
def describe_pairs(N=4):
    print(f'Batch={N}  Total views=2N={2*N}')
    print(f'  View-1 indices: 0 ... {N-1}')
    print(f'  View-2 indices: {N} ... {2*N-1}\n')
    print(f'  {"Original":<16} {"View-1 idx":>12} {"View-2 idx":>12} {"Positive Pair":>14}')
    print('  ' + '-'*56)
    for i in range(N):
        print(f'  {"image "+str(i):<16} {i:>12} {i+N:>12} {"yes":>14}')
    print('\n  All other (i,k) where k!=i and k!=i+N  ->  NEGATIVE pairs\n')

describe_pairs(4)

## 2.4 — Cosine Similarity Matrix (2N x 2N)

In [ ]:
def cosine_similarity_matrix(z1, z2):
    """
    Build (2N x 2N) cosine similarity matrix.
    1. Z = concat([z1, z2])  ->  (2N, D)
    2. L2-normalise rows
    3. Return Z @ Z.T
    """
    Z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)   # (2N, D)
    return Z @ Z.T                                         # (2N, 2N)

# Test
S = cosine_similarity_matrix(torch.randn(4,128), torch.randn(4,128))
print('Shape:', S.shape, '  diag max (should be ~1.0):', S.diag().max().item())

## 2.5 — NT-Xent Loss (from scratch — no library)

In [ ]:
class NTXentLoss(nn.Module):
    """
    Normalised Temperature-scaled Cross Entropy Loss.

    For anchor z_i, the positive is z_{i+N} (or z_{i-N}).
    Loss for z_i:
        l(i,j) = -log[ exp(sim(z_i,z_j)/tau) / sum_{k!=i} exp(sim(z_i,z_k)/tau) ]
    Total = mean over all 2N anchors.
    """
    def __init__(self, tau=0.5):
        super().__init__()
        self.tau = tau

    def forward(self, z1, z2):
        N      = z1.size(0)
        device = z1.device

        # Step 1: (2N x 2N) similarity / temperature
        sim = cosine_similarity_matrix(z1, z2) / self.tau

        # Step 2: mask diagonal (self-similarity)
        sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=device), float('-inf'))

        # Step 3: positive-pair labels
        labels = torch.cat([
            torch.arange(N, 2*N, device=device),   # rows 0..N-1  -> partner at N..2N-1
            torch.arange(0, N,   device=device),   # rows N..2N-1 -> partner at 0..N-1
        ])

        # Step 4: cross-entropy over 2N views
        return F.cross_entropy(sim, labels)

# Sanity check
crit_ntx = NTXentLoss(tau=TAU)
loss_rnd  = crit_ntx(torch.randn(8,128), torch.randn(8,128)).item()
expected  = -np.log(1/(2*8 - 1))
print(f'NT-Xent (random): {loss_rnd:.4f}  Expected~{expected:.4f}')

## 2.6 — Similarity Heatmap BEFORE Training

In [ ]:
def plot_sim_heatmap(model, loader, device, out_path, title, n=8):
    model.eval()
    with torch.no_grad():
        v1, v2, _ = next(iter(loader))
        z1, z2    = model(v1[:n].to(device), v2[:n].to(device))
    sim = cosine_similarity_matrix(z1, z2).cpu().numpy()

    fig, ax = plt.subplots(figsize=(8,7))
    im = ax.imshow(sim, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axhline(n-0.5, color='k', lw=1.5, ls='--')
    ax.axvline(n-0.5, color='k', lw=1.5, ls='--')
    tl = [f'v1_{i}' for i in range(n)] + [f'v2_{i}' for i in range(n)]
    ax.set_xticks(range(2*n)); ax.set_xticklabels(tl, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(2*n)); ax.set_yticklabels(tl, fontsize=7)
    ax.set_title(title, fontsize=11, pad=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show(); print(f'Saved -> {out_path}')


@torch.no_grad()
def sim_stats(model, loader, device, nb=10):
    model.eval(); same, diff = [], []
    for i, (v1, v2, _) in enumerate(loader):
        if i >= nb: break
        v1, v2 = v1.to(device), v2.to(device)
        N  = v1.size(0)
        z1 = F.normalize(model.encoder(v1), dim=1)
        z2 = F.normalize(model.encoder(v2), dim=1)
        S  = z1 @ z2.T
        same.extend(S.diag().cpu().tolist())
        diff.extend(S[~torch.eye(N, dtype=torch.bool, device=device)].cpu().tolist())
    return sum(same)/len(same), sum(diff)/len(diff)

In [ ]:
set_seed(SEED)
simclr_model = SimCLR().to(device)

# DataLoader using labeled split (for CP2 visualisation only)
base_raw = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_labeled_10percent.txt',
                              train=True, transform=None)
tv_ds    = TwoViewDataset(base_raw, TwoViewTransform(simclr_transform))
tv_ldr   = DataLoader(tv_ds, batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)

sb, db = sim_stats(simclr_model, tv_ldr, device)
print(f'BEFORE training:  same-image={sb:.4f}  diff-image={db:.4f}')

plot_sim_heatmap(
    simclr_model, tv_ldr, device,
    f'{RESULTS}/similarity_matrix_before_training.png',
    'Cosine Similarity Matrix — Before SimCLR Training (Random Encoder)',
)

---
# CHECKPOINT 3 — Day 9
> SimCLR Pretraining · Loss Curve · Similarity Before vs After

## 3.1 — Unlabelled SSL DataLoader

In [ ]:
ssl_base = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_ssl_unlabeled.txt',
                              train=True, transform=None, download=True)
ssl_tv   = TwoViewDataset(ssl_base, TwoViewTransform(simclr_transform))
ssl_ldr  = DataLoader(ssl_tv, batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
print(f'SSL unlabelled set: {len(ssl_base):,}  |  Batches per epoch: {len(ssl_ldr)}')

## 3.2 — SimCLR Pre-training (50 epochs)

In [ ]:
EPOCHS_SSL = 50
set_seed(SEED)
simclr_model = SimCLR().to(device)
ssl_crit     = NTXentLoss(tau=TAU)
ssl_opt      = torch.optim.Adam(simclr_model.parameters(), lr=LR)

ssl_losses = []
print(f'SimCLR pre-training  ({EPOCHS_SSL} epochs, labels NOT used) ...')
for ep in range(1, EPOCHS_SSL + 1):
    simclr_model.train()
    ep_loss, nb = 0.0, 0
    for v1, v2, _ in ssl_ldr:
        v1, v2 = v1.to(device), v2.to(device)
        ssl_opt.zero_grad()
        z1, z2 = simclr_model(v1, v2)
        loss   = ssl_crit(z1, z2)
        loss.backward(); ssl_opt.step()
        ep_loss += loss.item(); nb += 1
    avg = ep_loss / nb
    ssl_losses.append(avg)
    if ep % 5 == 0 or ep == 1:
        print(f'  Ep {ep:3d}/{EPOCHS_SSL}  NT-Xent={avg:.4f}')
print('Pre-training complete.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, EPOCHS_SSL+1), ssl_losses, lw=2, label='NT-Xent Loss')
ax.set(xlabel='Epoch', ylabel='NT-Xent Loss', title='SimCLR Pre-training Loss Curve')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{GRAPHS}/simclr_pretraining_loss.png', dpi=150)
plt.show(); print(f'Saved -> {GRAPHS}/simclr_pretraining_loss.png')

## 3.3 — Similarity Before vs After Training

In [ ]:
sa, da = sim_stats(simclr_model, ssl_ldr, device)
plot_sim_heatmap(
    simclr_model, ssl_ldr, device,
    f'{RESULTS}/similarity_matrix_after_training.png',
    'Cosine Similarity Matrix — After SimCLR Training',
)

print('\nFeature Similarity Comparison:')
print(f'{"Pair Type":<42} {"Before":>8} {"After":>8}')
print('-'*60)
print(f'{"Same image, two augmented views":<42} {sb:>8.4f} {sa:>8.4f}')
print(f'{"Different images":<42} {db:>8.4f} {da:>8.4f}')
print('\nSame-image similarity should increase after SimCLR training.')

torch.save(simclr_model.encoder.state_dict(), f'{MODELS}/simclr_encoder.pt')
print(f'\nSaved -> {MODELS}/simclr_encoder.pt')

---
# CHECKPOINT 4 — Day 12
> Linear Probe · Fine-tuning · PCA/t-SNE · metrics.json · test_predictions.csv

## 4.1 — Linear Probe (Frozen Encoder)

In [ ]:
# Feature extraction and linear probe helpers

@torch.no_grad()
def extract_features(encoder, loader, device):
    encoder.eval()
    feats, labs = [], []
    for x, y in loader:
        feats.append(encoder(x.to(device)).cpu())
        labs.append(y)
    return torch.cat(feats), torch.cat(labs)


def train_linear_head(linear, Xtr, ytr, Xvl, yvl, device, epochs=20, lr=3e-4):
    Xtr, ytr = Xtr.to(device), ytr.to(device)
    Xvl, yvl = Xvl.to(device), yvl.to(device)
    lc   = nn.CrossEntropyLoss()
    opt  = torch.optim.Adam(linear.parameters(), lr=lr)
    tr_a, vl_a = [], []
    for ep in range(1, epochs + 1):
        linear.train()
        opt.zero_grad()
        out  = linear(Xtr)
        lc(out, ytr).backward(); opt.step()
        ta = (out.argmax(1) == ytr).float().mean().item()
        linear.eval()
        with torch.no_grad():
            va = (linear(Xvl).argmax(1) == yvl).float().mean().item()
        tr_a.append(ta); vl_a.append(va)
        if ep % 5 == 0 or ep == 1:
            print(f'  Ep {ep:3d}  tr_acc={ta:.3f}  vl_acc={va:.3f}')
    return tr_a, vl_a


# Labeled loaders (eval transform, no augmentation)
ef = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
kw_ev = dict(batch_size=BATCH, num_workers=2, pin_memory=True, shuffle=False)
tr_ldr_ev = DataLoader(get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_labeled_10percent.txt', train=True,  transform=ef), **kw_ev)
vl_ldr_ev = DataLoader(get_cifar10_subset(DATA_ROOT, f'{SPLITS}/val.txt',  train=True,  transform=ef), **kw_ev)
te_ldr_ev = DataLoader(get_cifar10_subset(DATA_ROOT, f'{SPLITS}/test.txt', train=False, transform=ef), **kw_ev)
print('Loaders ready.')

In [ ]:
# Experiment A: RANDOM frozen encoder
print('='*55)
print('Experiment A: Random frozen encoder + linear head')
print('='*55)
enc_A = Encoder().to(device)   # no weights loaded
Xtr_A, ytr_A = extract_features(enc_A, tr_ldr_ev, device)
Xvl_A, yvl_A = extract_features(enc_A, vl_ldr_ev, device)
Xte_A, yte_A = extract_features(enc_A, te_ldr_ev, device)

LEPOCHS = 20
lin_A = nn.Linear(512, 10).to(device)
trA, vlA = train_linear_head(lin_A, Xtr_A, ytr_A, Xvl_A, yvl_A, device, LEPOCHS)
lin_A.eval()
with torch.no_grad():
    acc_A = (lin_A(Xte_A.to(device)).argmax(1) == yte_A.to(device)).float().mean().item()
print(f'[A] Random linear probe test acc: {acc_A:.4f}')

In [ ]:
# Experiment B: SIMCLR frozen encoder
print('='*55)
print('Experiment B: SimCLR frozen encoder + linear head')
print('='*55)
enc_B = Encoder().to(device)
enc_B.load_state_dict(torch.load(f'{MODELS}/simclr_encoder.pt', map_location=device))
Xtr_B, ytr_B = extract_features(enc_B, tr_ldr_ev, device)
Xvl_B, yvl_B = extract_features(enc_B, vl_ldr_ev, device)
Xte_B, yte_B = extract_features(enc_B, te_ldr_ev, device)

lin_B = nn.Linear(512, 10).to(device)
trB, vlB = train_linear_head(lin_B, Xtr_B, ytr_B, Xvl_B, yvl_B, device, LEPOCHS)
lin_B.eval()
with torch.no_grad():
    acc_B = (lin_B(Xte_B.to(device)).argmax(1) == yte_B.to(device)).float().mean().item()
print(f'[B] SimCLR linear probe test acc: {acc_B:.4f}')
torch.save(lin_B.state_dict(), f'{MODELS}/linear_probe.pt')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, ta, va, acc) in zip(axes, [
    ('Random', trA, vlA, acc_A),
    ('SimCLR', trB, vlB, acc_B)]):
    ep_r = range(1, LEPOCHS+1)
    ax.plot(ep_r, ta, lw=2, label='Train Acc')
    ax.plot(ep_r, va, lw=2, label='Val Acc')
    ax.set_title(f'{label} Encoder (test={acc:.3f})', fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle('Linear Probe Accuracy — Random vs SimCLR Encoder', fontsize=13)
fig.tight_layout()
fig.savefig(f'{GRAPHS}/linear_probe_accuracy.png', dpi=150)
plt.show(); print(f'Saved -> {GRAPHS}/linear_probe_accuracy.png')

## 4.2 — Fine-tuning (End-to-End)

In [ ]:
class FineTuneModel(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.head(self.encoder(x))

# Load SimCLR encoder, then fine-tune everything end-to-end
set_seed(SEED)
ft_tr_tf = T.Compose([T.RandomCrop(32,padding=4), T.RandomHorizontalFlip(),
                       T.ToTensor(), T.Normalize(MEAN, STD)])
ft_tr_ds  = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/train_labeled_10percent.txt',
                               train=True, transform=ft_tr_tf)
ft_tr_ldr = DataLoader(ft_tr_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

enc_ft = Encoder()
enc_ft.load_state_dict(torch.load(f'{MODELS}/simclr_encoder.pt', map_location='cpu'))
ft_model = FineTuneModel(enc_ft).to(device)
ft_crit  = nn.CrossEntropyLoss()
ft_opt   = torch.optim.Adam(ft_model.parameters(), lr=LR)

FEPOCHS = 20
ft_tra, ft_vla = [], []
best_ft, best_ft_sd = 0.0, None
print(f'Fine-tuning ({FEPOCHS} epochs) ...')
for ep in range(1, FEPOCHS+1):
    _, ta        = train_one_epoch(ft_model, ft_tr_ldr, ft_crit, ft_opt, device)
    _, va, _, _  = eval_epoch(ft_model, vl_ldr, ft_crit, device)
    ft_tra.append(ta); ft_vla.append(va)
    if va > best_ft:
        best_ft    = va
        best_ft_sd = {k: v.clone() for k, v in ft_model.state_dict().items()}
    if ep % 5 == 0 or ep == 1:
        print(f'  Ep {ep:3d}  tr={ta:.3f}  vl={va:.3f}')

ft_model.load_state_dict(best_ft_sd)
_, ft_acc, ft_preds, ft_labs = eval_epoch(ft_model, te_ldr, ft_crit, device)
print(f'Fine-tune Test Accuracy: {ft_acc:.4f}')
torch.save(ft_model.state_dict(), f'{MODELS}/finetuned_model.pt')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, FEPOCHS+1), ft_tra, lw=2, label='Train Acc')
ax.plot(range(1, FEPOCHS+1), ft_vla, lw=2, label='Val Acc')
ax.set_title(f'SimCLR Fine-tuning Accuracy  (test={ft_acc:.3f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(f'{GRAPHS}/finetuning_accuracy.png', dpi=150)
plt.show(); print(f'Saved -> {GRAPHS}/finetuning_accuracy.png')

## 4.3 — PCA / t-SNE Visualisation (1000 val images)

In [ ]:
from sklearn.manifold import TSNE
from torch.utils.data import Subset

N_VIS = 1000
set_seed(SEED)
rng      = np.random.default_rng(SEED)
val_full = get_cifar10_subset(DATA_ROOT, f'{SPLITS}/val.txt', train=True, transform=ef)
idx      = rng.choice(len(val_full), size=N_VIS, replace=False)
val_sub  = Subset(val_full, idx)
vis_ldr  = DataLoader(val_sub, batch_size=256, shuffle=False, num_workers=2)

@torch.no_grad()
def get_feats(enc, loader, device):
    enc.eval(); feats, labs = [], []
    for x, y in loader:
        feats.append(enc(x.to(device)).cpu())
        labs.append(y)
    return torch.cat(feats).numpy(), torch.cat(labs).numpy()


def plot_tsne(feats, labels, out_path, title):
    set_seed(SEED)
    reduced = TSNE(2, random_state=SEED, perplexity=30,
                   n_iter=1000, init='pca').fit_transform(feats)
    pal = plt.cm.get_cmap('tab10', 10)
    fig, ax = plt.subplots(figsize=(9, 8))
    for ci, cname in enumerate(CLASSES):
        mask = labels == ci
        ax.scatter(reduced[mask,0], reduced[mask,1],
                   s=10, alpha=0.65, color=pal(ci), label=cname)
    ax.legend(markerscale=2.5, fontsize=9, ncol=2, framealpha=0.8)
    ax.set_title(f'{title}\n(t-SNE, {N_VIS} val images)', fontsize=12)
    ax.grid(alpha=0.2); fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show(); print(f'Saved -> {out_path}')

In [ ]:
# 1. Random encoder
enc_rnd = Encoder().to(device)
feats_r, vis_labels = get_feats(enc_rnd, vis_ldr, device)
plot_tsne(feats_r, vis_labels,
          f'{RESULTS}/random_encoder_pca_or_tsne.png', 'Random (Untrained) Encoder')

# 2. SimCLR encoder
enc_s = Encoder().to(device)
enc_s.load_state_dict(torch.load(f'{MODELS}/simclr_encoder.pt', map_location=device))
feats_s, _ = get_feats(enc_s, vis_ldr, device)
plot_tsne(feats_s, vis_labels,
          f'{RESULTS}/simclr_encoder_pca_or_tsne.png', 'SimCLR Pre-trained Encoder')

# 3. Fine-tuned encoder
feats_f, _ = get_feats(ft_model.encoder, vis_ldr, device)
plot_tsne(feats_f, vis_labels,
          f'{RESULTS}/finetuned_encoder_pca_or_tsne.png', 'Fine-tuned Encoder')

print('\nt-SNE answers:')
print('Q1. Random: no class grouping — points scattered randomly.')
print('Q2. SimCLR: visible clusters — self-supervised training helps grouping.')
print('Q3. Fine-tuned: sharpest separation — supervised signal sharpens boundaries.')
print('Q4. Still confused: cat/dog, automobile/truck, bird/airplane.')

## 4.4 — metrics.json & test_predictions.csv

In [ ]:
import json, csv

metrics = {
    'student_name':                    'YourName',
    'roll_number':                     'YourRollNumber',
    'seed':                            SEED,
    'batch_size':                      BATCH,
    'simclr_epochs':                   EPOCHS_SSL,
    'linear_probe_epochs':             LEPOCHS,
    'finetuning_epochs':               FEPOCHS,
    'learning_rate':                   LR,
    'temperature':                     TAU,
    'supervised_10percent_test_acc':   round(sup_test_acc, 4),
    'random_linear_probe_test_acc':    round(acc_A, 4),
    'simclr_linear_probe_test_acc':    round(acc_B, 4),
    'simclr_finetune_test_acc':        round(ft_acc, 4),
    'same_view_similarity_before':     round(sb, 4),
    'different_image_similarity_before': round(db, 4),
    'same_view_similarity_after':      round(sa, 4),
    'different_image_similarity_after':  round(da, 4),
}

with open(f'{RESULTS}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('metrics.json:')
print(json.dumps(metrics, indent=2))

In [ ]:
# Compute per-sample probabilities for fine-tuned model
@torch.no_grad()
def get_probs_all(model, loader, device):
    model.eval()
    all_p, all_l, all_pr = [], [], []
    for x, y in loader:
        logits = model(x.to(device))
        pr     = F.softmax(logits, dim=1)
        all_p.extend(pr.argmax(1).cpu().tolist())
        all_l.extend(y.tolist())
        all_pr.extend(pr.cpu().tolist())
    return all_p, all_l, all_pr

ft_preds2, ft_labs2, ft_probs2 = get_probs_all(ft_model, te_ldr, device)

csv_path = f'{RESULTS}/test_predictions.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['image_index','true_label','predicted_label'] +
               [f'prob_class_{i}' for i in range(10)])
    for i, (p, l, pr) in enumerate(zip(ft_preds2, ft_labs2, ft_probs2)):
        w.writerow([i, l, p] + [round(x, 6) for x in pr])
print(f'Saved -> {csv_path}')

## 4.5 — Final Comparison Table

In [ ]:
print('='*78)
print('FINAL RESULTS')
print('='*78)
print(f'{"Model":<52} {"Labels":>7} {"Frozen":>7} {"Test Acc":>10}')
print('-'*78)
for name, lbl, frz, acc in [
    ('Supervised ResNet-18 (10% labels)',         'Yes', 'No',  sup_test_acc),
    ('Random encoder + linear probe',             'No',  'Yes', acc_A),
    ('SimCLR encoder + linear probe',             'No',  'Yes', acc_B),
    ('SimCLR encoder + full fine-tuning',         'Mix', 'No',  ft_acc),
]:
    print(f'  {name:<50} {lbl:>7} {frz:>7} {acc:>10.4f}')
print('='*78)

## 4.6 — List all generated files & download

In [ ]:
import os
print('Generated files:')
for folder in [GRAPHS, RESULTS, MODELS]:
    for fname in sorted(os.listdir(folder)):
        size = os.path.getsize(f'{folder}/{fname}')
        print(f'  {folder}/{fname:<45}  {size/1024:>8.1f} KB')

In [ ]:
import shutil
shutil.make_archive('SimCLR_Submission', 'zip', '.', '.')
from google.colab import files
files.download('SimCLR_Submission.zip')
print('Download started.')